In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "cinedata_analytics"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER} COMMENT 'Camada Silver - dados limpos, tipados e em português'")

def ler_bronze(tabela):
    """Leitura da Bronze (somente leitura: nenhuma alteração é feita na camada Bronze)."""
    return spark.table(f"{BRONZE}.{tabela}")

def ler_silver(tabela):
    return spark.table(f"{SILVER}.{tabela}")

def salvar_silver(df, tabela):
    """A Silver é sempre reconstruída a partir da Bronze completa (overwrite).
    Como a Bronze é append (histórico), reprocessar tudo mantém a Silver consistente e idempotente."""
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{SILVER}.{tabela}"))
    print(f"OK  {SILVER}.{tabela}: {ler_silver(tabela).count():,} linhas")

def try_cast(coluna, tipo):
    """Conversão segura: valores incompatíveis viram NULL em vez de quebrar o pipeline.
    Necessário porque o Serverless roda com ANSI mode ligado, e um cast() comum lançaria erro."""
    return F.expr(f"try_cast(`{coluna}` AS {tipo})")

def manter_mais_recente(df, chave, colunas_completude):
    """Deduplicação: mantém 1 registro por chave.
    Critério 1: ingestion_datetime mais recente (regra do escopo).
    Critério 2 (desempate): a base tem duplicatas DENTRO da mesma carga, com versões diferentes
    (ex.: uma com receita preenchida e outra 'Unknown'). Nesse empate, fica a versão mais completa."""
    completude = sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in colunas_completude)
    janela = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc(), completude.desc())
    return (df.withColumn("_rn", F.row_number().over(janela))
              .filter("_rn = 1")
              .drop("_rn"))

def ultima_carga_por_filme(df, chave="id"):
    """Para tabelas que serão 'explodidas' (gêneros, pessoas): mantém apenas as linhas
    da carga mais recente de cada filme, evitando misturar versões antigas da Bronze."""
    janela = Window.partitionBy(chave)
    return (df.withColumn("_max_ing", F.max("ingestion_datetime").over(janela))
              .filter(F.col("ingestion_datetime") == F.col("_max_ing"))
              .drop("_max_ing"))

# Textos que representam ausência de dado na origem (comparados em maiúsculas)
TOKENS_NULOS = ["", "UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "N/A", "NA", "NULL", "NONE",
                "NAN", "NENHUM", "[]", "-", "--"]

In [0]:
# Regras:
# - Converter dataHoraCotacao em DATE e deduplicar por dia (a Bronze é append, então o mesmo
#   dia aparece a cada carga). Fica o boletim mais recente do dia, da carga mais recente.
# - Gerar um calendário contínuo (todos os dias corridos) do primeiro dia com cotação até
#   o maior valor entre a última cotação e a data de execução. Assim fins de semana,
#   feriados e dias posteriores ao último boletim também têm valor.
# - Forward Fill: last(..., ignorenulls=True) em janela ordenada por data até a linha atual,
#   ou seja, cada dia sem cotação recebe o valor do último dia útil disponível.

df_cot_bronze = (ler_bronze("tb_cotacao_dolar")
    .withColumn("data_cotacao", F.to_date(F.try_to_timestamp(F.substring("dataHoraCotacao", 1, 10), F.lit("yyyy-MM-dd"))))
    .withColumn("cotacao_compra", try_cast("cotacaoCompra", "DECIMAL(10,4)"))
    .filter(F.col("data_cotacao").isNotNull() & (F.col("cotacao_compra") > 0)))

janela_dia = Window.partitionBy("data_cotacao").orderBy(F.col("ingestion_datetime").desc(),
                                                        F.col("dataHoraCotacao").desc())
df_cot_dia = (df_cot_bronze
    .withColumn("_rn", F.row_number().over(janela_dia))
    .filter("_rn = 1")
    .select("data_cotacao", "cotacao_compra"))

# Calendário contínuo
df_calendario = (df_cot_dia
    .agg(F.min("data_cotacao").alias("inicio"),
         F.greatest(F.max("data_cotacao"), F.current_date()).alias("fim"))
    .select(F.explode(F.sequence("inicio", "fim", F.expr("INTERVAL 1 DAY"))).alias("data_cotacao")))

# partitionBy(lit(1)): a série inteira forma uma única partição de forma explícita (é o necessário
# para o forward fill olhar todos os dias anteriores). A série tem poucos dias, então não há custo
# de performance, e deixar isso explícito evita o aviso de "No Partition Defined".
janela_ffill = (Window.partitionBy(F.lit(1)).orderBy("data_cotacao")
                      .rowsBetween(Window.unboundedPreceding, Window.currentRow))

df_cotacao_silver = (df_calendario
    .join(df_cot_dia, "data_cotacao", "left")
    .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())   # True = valor herdado (forward fill)
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_ffill))
    .select("data_cotacao", "cotacao_compra", "cotacao_preenchida")
    .orderBy("data_cotacao"))

salvar_silver(df_cotacao_silver, "tb_cotacao_dolar")
display(ler_silver("tb_cotacao_dolar").orderBy("data_cotacao"))

OK  cinedata_analytics.silver.tb_cotacao_dolar: 8 linhas


data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-11,5.0912,false
2026-09-12,5.0912,true
2026-09-13,5.0912,true
2026-09-14,5.1690,false
2026-09-15,5.1484,false
2026-09-16,5.1520,false
2026-09-17,5.1515,false
2026-09-18,5.1515,true


In [0]:
# --- Status: normalizar ANTES de traduzir ---
# Maiúsculas + hífen/underscore viram espaço + remoção de ruído (qualquer caractere que não seja letra)
# + espaços colapsados. Assim "In-Production", "IN PRODUCTION" e "in production" viram "IN PRODUCTION".
status_norm = F.trim(F.regexp_replace(
                  F.regexp_replace(F.upper(F.col("status")), r"[^A-Z]+", " "),
                  r"\s+", " "))

TRADUCAO_STATUS = {
    "RELEASED":        "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION":   "Em Produção",
    "PLANNED":         "Planejado",
    "RUMORED":         "Rumores",
    "CANCELED":        "Cancelado",
    "CANCELLED":       "Cancelado",   # grafia britânica, por robustez
}
expr_status = F.lit("Não Informado")          # corrompidos / não mapeáveis (ex.: datas vazadas por column shift)
for origem, destino in TRADUCAO_STATUS.items():
    expr_status = F.when(status_norm == origem, destino).otherwise(expr_status)

# --- Data multi-formato ---
# Formatos confirmados na base: AAAA-MM-DD (ISO), DD/MM/AAAA (barra: 1º número chega a 31) e
# MM-DD-AAAA (hífen: 2º número chega a 31). Cada formato é identificado pelo padrão (regex) antes
# de converter, e try_to_timestamp devolve NULL em datas impossíveis (ex.: 30/02) em vez de quebrar.
# NULL só quando nenhum formato converte.
data_txt = F.trim(F.col("release_date"))
def converter(regex, formato):
    return F.when(data_txt.rlike(regex), F.to_date(F.try_to_timestamp(data_txt, F.lit(formato))))

expr_data = F.coalesce(
    converter(r"^\d{4}-\d{1,2}-\d{1,2}$", "yyyy-M-d"),   # 2016-02-09
    converter(r"^\d{1,2}/\d{1,2}/\d{4}$", "d/M/yyyy"),   # 25/12/2019
    converter(r"^\d{1,2}-\d{1,2}-\d{4}$", "M-d-yyyy"),   # 04-25-2018
)

# --- Título: alguns vêm TODO EM MAIÚSCULAS ou minúsculas (ex.: "AVENGERS: INFINITY WAR", "spider-man: no way home").
# Quando o título original é o mesmo texto com a capitalização correta, usamos a grafia do original.
titulo_corrigido = (F.when(((F.col("title") == F.upper("title")) | (F.col("title") == F.lower("title"))) &
                           (F.upper("title") != F.lower("title")) &
                           (F.lower("title") == F.lower("original_title")), F.col("original_title"))
                     .otherwise(F.col("title")))

df_info = (ler_bronze("tb_movies_info")
    .withColumn("id_filme", F.trim("id"))
    .filter(F.col("id_filme").rlike(r"^\d+$"))                    # descarta ids corrompidos
    .withColumn("_runtime", try_cast("runtime", "DOUBLE"))
    .select(
        "id_filme",
        F.trim(titulo_corrigido).alias("titulo"),
        F.trim("original_title").alias("titulo_original"),
        expr_data.alias("data_lancamento"),
        # duração 0 ou negativa na origem significa "desconhecida" → NULL
        F.when(F.col("_runtime") > 0, F.col("_runtime").cast("INT")).alias("duracao_minutos"),
        F.lower(F.trim("original_language")).alias("idioma_original"),
        expr_status.alias("status_filme"),
        F.trim("overview").alias("sinopse"),
        F.trim("tagline").alias("frase_divulgacao"),
        "ingestion_datetime",
    ))

df_info = (manter_mais_recente(df_info, "id_filme",
                ["titulo", "data_lancamento", "duracao_minutos", "sinopse", "frase_divulgacao"])
           .withColumn("ano_lancamento", F.year("data_lancamento"))    # coluna derivada
           .select("id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
                   "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao"))

salvar_silver(df_info, "tb_info_filmes")

OK  cinedata_analytics.silver.tb_info_filmes: 97,611 linhas


In [0]:
# Validações da tb_info_filmes
tb = ler_silver("tb_info_filmes")
display(tb.limit(10))
display(tb.groupBy("status_filme").count().orderBy(F.desc("count")))
display(tb.agg(F.count("*").alias("filmes"),
               F.countDistinct("id_filme").alias("ids_unicos"),                 # deve ser igual a "filmes"
               F.sum(F.col("data_lancamento").isNull().cast("int")).alias("datas_nulas")))

id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null
1000007,Kyle Brownrigg: Introducing Lyle,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,null,es,Lançado,null,null
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null
1000079,Pourquoi tu souris ?,Pourquoi tu souris ?,2024-06-26,2024,null,fr,Em Produção,null,null
1000081,Sentinelle,Sentinelle,2023-08-27,2023,98,fr,Lançado,"François Sentinelle has two lives. By day, he is the most famous cop of Réunion Island, known for his tough methods and flowery shirts, pursuing criminals in his famous yellow defender. But the rest of the time, Sentinelle is also a charming singer.",null
1000091,Amy Miller: Ham Mouth,Amy Miller: Ham Mouth,2022-03-24,2022,35,en,Lançado,"Amy Miller reflects on her recent breakup, shares her love of baths and reveals the 40-year-old shit she does.",null
1000094,Le clan,Le clan,2023-01-18,2023,90,fr,Lançado,"Fred, Achille, Max and Belette form a good-for-nothing gang of crooks. Dismally failing their last raid, they decide to get back in the game, kidnapping Sophie Marceau.",null


status_filme,count
Lançado,96261
Pós-Produção,697
Em Produção,604
Planejado,47
Não Informado,2


filmes,ids_unicos,datas_nulas
97611,97611,2


In [0]:
# 2) silver.tb_financeiro_filmes

def limpar_monetario(df, origem, destino):
    """Higieniza um valor monetário textual e converte para DECIMAL(18,2):
    1. Tokens de ausência ('Unknown', 'Não Informado', 'N/A'...) → NULL antes de qualquer conversão.
    2. Remove símbolos/códigos de moeda ('$', 'USD') e separadores de milhar (vírgula) e espaços.
    3. Expande abreviações: K = mil, M = milhão, B = bilhão (ex.: '34.0M' → 34000000).
       Remover só a letra transformaria 34.0M em 34, um erro de 6 ordens de grandeza.
    4. Zero ou negativo → NULL (a regra de negócio trata como dado ausente)."""
    txt = F.upper(F.trim(F.col(origem)))
    txt = F.when(txt.isin(TOKENS_NULOS), None).otherwise(txt)
    txt = F.regexp_replace(txt, r"US\$|R\$|USD|BRL|\$", "")
    txt = F.regexp_replace(txt, r"[\s,]", "")                       # remove espaços e vírgulas de milhar
    # Ponto como separador de milhar (ex.: '1.500', '1.000.000'): valores monetários têm no máximo
    # 2 casas decimais, então grupos de exatamente 3 dígitos após o ponto só podem ser milhar.
    # Números com sufixo ('34.0M', '99.9K') e decimais comuns ('1500000.50') não casam com o padrão.
    txt = F.when(txt.rlike(r"^-?\d{1,3}(\.\d{3})+$"), F.regexp_replace(txt, r"\.", "")).otherwise(txt)
    padrao = r"^(-?\d+(\.\d+)?)([KMB]?)$"
    sufixo = F.regexp_extract(txt, padrao, 3)
    multiplicador = (F.when(sufixo == "K", 1_000).when(sufixo == "M", 1_000_000)
                      .when(sufixo == "B", 1_000_000_000).otherwise(1))
    return (df
        .withColumn("_num", F.regexp_extract(txt, padrao, 1))          # '' quando não bate com o padrão
        .withColumn("_valor", try_cast("_num", "DECIMAL(20,2)") * multiplicador)
        .withColumn("_valor", try_cast("_valor", "DECIMAL(18,2)"))
        .withColumn(destino, F.when(F.col("_valor") > 0, F.col("_valor")))
        .drop("_num", "_valor"))

df_fin = ler_bronze("tb_movies_financials").withColumn("id_filme", F.trim("id")).filter(F.col("id_filme").rlike(r"^\d+$"))
df_fin = limpar_monetario(df_fin, "budget",  "orcamento_usd")
df_fin = limpar_monetario(df_fin, "revenue", "receita_usd")
df_fin = manter_mais_recente(df_fin, "id_filme", ["orcamento_usd", "receita_usd"])

# --- Cotação: os filmes não têm data de transação e a série PTAX cobre só os últimos dias,
# então um join pela data de lançamento não encontraria cotação para quase nenhum filme.
# Aplicamos a cotação mais recente disponível (valor atual em reais, como pede a diretoria financeira).
# A data da cotação usada fica gravada para rastreabilidade.
df_cot_atual = (ler_silver("tb_cotacao_dolar")
    .orderBy(F.col("data_cotacao").desc()).limit(1)
    .select(F.col("data_cotacao").alias("data_cotacao_utilizada"),
            F.col("cotacao_compra").alias("cotacao_dolar_utilizada")))

df_fin = (df_fin.crossJoin(df_cot_atual)     # 1 linha só: não multiplica registros
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("cotacao_dolar_utilizada")).cast("DECIMAL(18,2)"))
    .withColumn("receita_brl",   (F.col("receita_usd")   * F.col("cotacao_dolar_utilizada")).cast("DECIMAL(18,2)"))
    # Lucro só existe se orçamento E receita forem conhecidos. Trocar nulo por 0 inventaria lucro
    # (receita sem custo) ou prejuízo (custo sem receita). A subtração com nulo devolve NULL sem erro.
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("DECIMAL(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("DECIMAL(18,2)"))
    # Margem (%) = lucro / receita * 100, com proteção explícita contra divisão por zero/nulo.
    # DOUBLE porque margens de filmes com receita ínfima podem ser muito grandes (ex.: -99.999%).
    .withColumn("margem_lucro_percentual",
                F.when(F.col("receita_usd") > 0,
                       F.round(F.try_divide(F.col("lucro_usd").cast("DOUBLE"), F.col("receita_usd").cast("DOUBLE")) * 100, 2)))
    .select("id_filme", "orcamento_usd", "receita_usd", "lucro_usd",
            "orcamento_brl", "receita_brl", "lucro_brl", "margem_lucro_percentual",
            "cotacao_dolar_utilizada", "data_cotacao_utilizada"))

salvar_silver(df_fin, "tb_financeiro_filmes")
tb = ler_silver("tb_financeiro_filmes")
display(tb.filter("receita_usd IS NOT NULL AND orcamento_usd IS NOT NULL").orderBy(F.desc("receita_usd")).limit(10))
display(tb.agg(F.count("*").alias("filmes"), F.countDistinct("id_filme").alias("ids_unicos"),
               F.count("orcamento_usd").alias("com_orcamento"), F.count("receita_usd").alias("com_receita"),
               F.count("lucro_usd").alias("com_lucro")))

OK  cinedata_analytics.silver.tb_financeiro_filmes: 99,006 linhas


id_filme,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,margem_lucro_percentual,cotacao_dolar_utilizada,data_cotacao_utilizada
299534,356000000.00,2800000000.00,2444000000.00,1833934000.00,14424200000.00,12590266000.00,87.29,5.1515,2026-09-18
76600,460000000.00,2320250281.00,1860250281.00,2369690000.00,11952769322.57,9583079322.57,80.17,5.1515,2026-09-18
299536,300000000.00,2052415039.00,1752415039.00,1545450000.00,10573016073.41,9027566073.41,85.38,5.1515,2026-09-18
634649,200000000.00,1921847111.00,1721847111.00,1030300000.00,9900395392.32,8870095392.32,89.59,5.1515,2026-09-18
420818,260000000.00,1663075401.00,1403075401.00,1339390000.00,8567332928.25,7227942928.25,84.37,5.1515,2026-09-18
361743,170000000.00,1488732821.00,1318732821.00,875755000.00,7669207127.38,6793452127.38,88.58,5.1515,2026-09-18
346698,145000000.00,1428545028.00,1283545028.00,746967500.00,7359149711.74,6612182211.74,89.85,5.1515,2026-09-18
502356,100000000.00,1355725263.00,1255725263.00,515150000.00,6984018692.34,6468868692.34,92.62,5.1515,2026-09-18
284054,200000000.00,1349926083.00,1149926083.00,1030300000.00,6954144216.57,5923844216.57,85.18,5.1515,2026-09-18
181808,200000000.00,1332698830.00,1132698830.00,1030300000.00,6865398022.75,5835098022.75,84.99,5.1515,2026-09-18


filmes,ids_unicos,com_orcamento,com_receita,com_lucro
99006,99006,8193,3298,1576


In [0]:
# 3) silver.tb_metricas_engajamento

# - Popularidade: a vírgula é usada como separador decimal em parte das linhas ("154,34").
#   Na base não há separador de milhar, então trocar vírgula por ponto preserva o valor.
#   Sem essa troca o valor viraria NULL silenciosamente.
# - Column shift: textos de sinopse/diretor vazaram para colunas numéricas. try_cast converte
#   o que é número e transforma o resto em NULL, sem interromper o pipeline.
#   Contagens usam INT direto, então resíduos fracionários deslocados (ex.: '0.6' em numVotes) também viram NULL.
# - Limites de negócio: notas fora de 0-10 (ex.: 79.97, erro de escala ×10) → NULL (não dividimos
#   por 10, porque não há como garantir que o fator seja sempre 10). Negativos em votos/popularidade → NULL.

df_met = (ler_bronze("tb_movies_metrics")
    .withColumn("id_filme", F.trim("id"))
    .filter(F.col("id_filme").rlike(r"^\d+$"))
    .withColumn("_pop",  F.regexp_replace(F.trim("popularity"), ",", "."))
    .withColumn("_pop",  try_cast("_pop", "DOUBLE"))
    .withColumn("_vavg", try_cast("vote_average", "DOUBLE"))
    .withColumn("_vcnt", try_cast("vote_count", "INT"))
    .withColumn("_avgr", try_cast("averageRating", "DOUBLE"))
    .withColumn("_nvot", try_cast("numVotes", "INT"))
    .select(
        "id_filme",
        # Column shift: anos vazados para a popularidade (ex.: 2020.0, 1969.0). Popularidade real do
        # TMDB tem casas decimais; inteiro exato entre 1870 e 2030 é ano deslocado → NULL.
        F.when((F.col("_pop") >= 0) &
               ~((F.col("_pop") == F.floor("_pop")) & F.col("_pop").between(1870, 2030)),
               F.col("_pop")).alias("popularidade"),
        F.when(F.col("_vavg").between(0, 10), F.col("_vavg")).alias("nota_media_tmdb"),
        F.when(F.col("_vcnt") >= 0, F.col("_vcnt")).alias("qtd_votos_tmdb"),
        F.when(F.col("_avgr").between(0, 10), F.col("_avgr")).alias("nota_media_imdb"),
        F.when(F.col("_nvot") >= 0, F.col("_nvot")).alias("qtd_votos_imdb"),
        "ingestion_datetime",
    ))

df_met = manter_mais_recente(df_met, "id_filme",
            ["popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"]
         ).drop("ingestion_datetime")

salvar_silver(df_met, "tb_metricas_engajamento")
tb = ler_silver("tb_metricas_engajamento")
display(tb.orderBy(F.desc("popularidade")).limit(10))
display(tb.agg(F.count("*").alias("filmes"), F.countDistinct("id_filme").alias("ids_unicos"),
               F.max("nota_media_tmdb").alias("max_nota_tmdb"), F.max("nota_media_imdb").alias("max_nota_imdb"),
               F.count("popularidade").alias("com_popularidade")))

OK  cinedata_analytics.silver.tb_metricas_engajamento: 95,115 linhas


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
565770,2994.357,7.139,1023,5.9,119501
980489,2680.593,8.068,null,7.1,138696
968051,1692.778,6.545,365,5.6,84018
615656,1567.273,6.912,2034,null,105031
762430,1547.22,6.816,258,null,29152
1008042,1458.514,7.214,973,7.1,224148
385687,1175.267,null,3881,5.7,149785
678512,1111.036,7.973,503,7.6,null
346698,1069.34,7.279,5074,6.8,704472
976573,1008.942,7.757,2467,7.0,169047


filmes,ids_unicos,max_nota_tmdb,max_nota_imdb,com_popularidade
95115,95115,10.0,10.0,91576


In [0]:
# 4) silver.tb_avaliacoes_usuarios

# - Nota fora da escala 0-10 → NULL (o registro é mantido, pois o comentário ainda tem valor).
# - Comentário nulo, vazio ou só com espaços → "Sem comentário" (preenchimento explícito).
# - Duplicatas: remove registros idênticos em filme + usuário + nota + comentário, APÓS a padronização.
#   Assim, "  ótimo " e "ótimo" contam como iguais, e as recargas da Bronze (append) não duplicam avaliações.

df_rev = (ler_bronze("tb_movies_reviews")
    .withColumn("id_filme", F.trim("id"))
    .filter(F.col("id_filme").rlike(r"^\d+$"))
    .withColumn("_nota", try_cast("nota", "DOUBLE"))
    .select(
        "id_filme",
        F.trim("nome").alias("nome_usuario"),
        F.when(F.col("_nota").between(0, 10), F.col("_nota")).alias("nota_usuario"),
        F.when(F.trim(F.coalesce(F.col("comentario"), F.lit(""))) == "", F.lit("Sem comentário"))
         .otherwise(F.trim("comentario")).alias("comentario_usuario"),
    )
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]))

salvar_silver(df_rev, "tb_avaliacoes_usuarios")
tb = ler_silver("tb_avaliacoes_usuarios")
display(tb.limit(10))
display(tb.agg(F.count("*").alias("avaliacoes"),
               F.sum((F.col("comentario_usuario") == "Sem comentário").cast("int")).alias("sem_comentario"),
               F.sum(F.col("nota_usuario").isNull().cast("int")).alias("notas_nulas"),
               F.min("nota_usuario").alias("nota_min"), F.max("nota_usuario").alias("nota_max")))

OK  cinedata_analytics.silver.tb_avaliacoes_usuarios: 32,412 linhas


id_filme,nome_usuario,nota_usuario,comentario_usuario
637007,Lucas Reis 602,3.9,Sem comentário
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
640543,Cristina Monteiro 310,6.4,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.


avaliacoes,sem_comentario,notas_nulas,nota_min,nota_max
32412,7776,1651,0.0,10.0


In [0]:
# 5) silver.tb_generos — confirmação dos separadores presentes na base real

df_credits = ultima_carga_por_filme(ler_bronze("tb_credits_and_tags").withColumn("id", F.trim("id")))

display(df_credits.agg(
    F.sum(F.col("genres").contains(",").cast("int")).alias("linhas_com_virgula"),
    F.sum(F.col("genres").contains(";").cast("int")).alias("linhas_com_ponto_e_virgula"),
    F.sum(F.col("genres").contains("|").cast("int")).alias("linhas_com_pipe"),
))

linhas_com_virgula,linhas_com_ponto_e_virgula,linhas_com_pipe
36042,3157,4257


In [0]:
# - Separadores: a base usa vírgula, ponto e vírgula E pipe ("Comedy|Drama"). Todos são
#   padronizados para vírgula antes do split.
# - Limpeza de resíduos: em vez de tentar prever todo tipo de lixo (números deslocados como "0.6",
#   trechos de texto, vazios), validamos contra o DOMÍNIO oficial de gêneros do TMDB (19 gêneros).
#   Só entra o que pertence ao domínio. A comparação ignora caixa e devolve a grafia canônica.
GENEROS_VALIDOS = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
                   "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
                   "Science Fiction", "TV Movie", "Thriller", "War", "Western"]
df_dominio = spark.createDataFrame([(g.lower(), g) for g in GENEROS_VALIDOS], ["_chave", "nome_genero"])

df_generos = (df_credits
    .filter(F.col("id").rlike(r"^\d+$"))
    .select(F.col("id").alias("id_filme"),
            F.explode(F.split(F.regexp_replace("genres", r"[;|]", ","), ",")).alias("_genero"))
    .withColumn("_chave", F.lower(F.regexp_replace(F.trim("_genero"), r"\s+", " ")))
    .join(F.broadcast(df_dominio), "_chave", "inner")       # descarta tudo que não é gênero
    .select("id_filme", "nome_genero")
    .dropDuplicates())

salvar_silver(df_generos, "tb_generos")
display(ler_silver("tb_generos").groupBy("nome_genero").count().orderBy(F.desc("count")))

OK  cinedata_analytics.silver.tb_generos: 141,941 linhas


nome_genero,count
Drama,32615
Documentary,19235
Comedy,18827
Thriller,10400
Horror,9833
Romance,7717
Action,6114
Crime,4786
Animation,4524
TV Movie,4115


In [0]:
# 6) silver.tb_pessoas_empresas

# 1. Unificação: as 4 colunas viram linhas com tipo_entidade (cast→Ator, directors→Diretor,
#    writers→Roteirista, production_companies→Produtora), e cada campo multivalorado é explodido.
# 2. Limpeza de resíduos do column shift: placeholders ('[]', 'Nenhum', 'N/A'), valores só numéricos,
#    textos longos demais para um nome (trechos de sinopse), barras/aspas de escape nas pontas,
#    nomes de gênero (ex.: 'Documentary' em produtoras) e de países/idiomas (ex.: 'English' em diretores)
#    que vazaram de outras colunas.
# 3. Capitalização: nomes de pessoas TODO EM MAIÚSCULAS ou minúsculas recebem initcap (produtoras só
#    quando minúsculas, para preservar siglas como BBC). Nomes já em caixa mista são preservados,
#    porque initcap estragaria grafias como 'McDonagh' ou 'DreamWorks'.
# 4. Grafia canônica: a mesma entidade pode aparecer como 'KEVIN HART' e 'Kevin Hart'. Para cada
#    nome (ignorando caixa) escolhemos a grafia em caixa mista mais frequente, para que a dimensão da Gold
#    não tenha a mesma pessoa duas vezes.

MAPA_ENTIDADES = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista",
                  "production_companies": "Produtora"}

df_entidades = None
for coluna, tipo in MAPA_ENTIDADES.items():
    parte = (df_credits
        .filter(F.col("id").rlike(r"^\d+$"))
        .select(F.col("id").alias("id_filme"),
                F.explode(F.split(F.col(coluna), r"[,;]")).alias("_nome"))
        .withColumn("tipo_entidade", F.lit(tipo)))
    df_entidades = parte if df_entidades is None else df_entidades.unionByName(parte)

generos_lower = [g.lower() for g in GENEROS_VALIDOS]

# Resíduos de país/idioma/keyword: o column shift empurra valores de production_countries,
# spoken_languages e keywords para directors/cast/writers (ex.: 'English', 'United States of America',
# 'Short Film' como "diretor").
# Porém essas colunas TAMBÉM estão contaminadas (há nomes de pessoas e produtoras nelas), então
# não dá para usá-las como lista negra direta. Regra por frequência: o valor só é descartado
# se aparece MAIS vezes como país/idioma do que como entidade.
# Ex.: 'English' = milhares como idioma vs. ~300 como diretor → descarta.
#      'Kevin Dunn' = 1 vez como "país" vs. 77 como diretor → mantém.
df_residuos = (df_credits
    .select(F.explode(F.split(F.concat_ws(",", "production_countries", "spoken_languages", "keywords"), r"[,;]")).alias("_r"))
    .select(F.lower(F.trim("_r")).alias("_chave"))
    .filter(F.length("_chave") > 1)
    .groupBy("_chave").agg(F.count("*").alias("_freq_pais_idioma")))

nome = F.regexp_replace(F.regexp_replace(F.col("_nome"), r'^[\s\\"\']+|[\s\\"\']+$', ""), r"\s+", " ")

df_entidades = (df_entidades
    .withColumn("_nome", nome)
    .filter(F.col("_nome").isNotNull()
            & (F.length("_nome").between(2, 100))
            & (~F.upper("_nome").isin(TOKENS_NULOS))
            & (~F.col("_nome").rlike(r"^[\d\s.,:;/%()+-]+$"))
            & (~F.lower("_nome").isin(generos_lower)))              # gênero vazado de coluna
    .withColumn("_caixa_mista", (F.col("_nome") != F.upper("_nome")) & (F.col("_nome") != F.lower("_nome")))
    # Pessoas: TUDO MAIÚSCULO ou minúsculo → initcap. Produtoras: só minúsculo → initcap, porque
    # siglas em maiúsculas são legítimas em nomes de empresas (BBC, ZDF, ARTE).
    .withColumn("_padronizar", ~F.col("_caixa_mista") &
                ((F.col("tipo_entidade") != "Produtora") | (F.col("_nome") == F.lower("_nome"))))
    .withColumn("_nome", F.when(F.col("_padronizar"), F.initcap("_nome")).otherwise(F.col("_nome")))
    .drop("_padronizar")
    .withColumn("_chave", F.lower("_nome"))
    .withColumn("_freq_entidade", F.count("*").over(Window.partitionBy("_chave")))
    .join(df_residuos, "_chave", "left")
    .filter(F.coalesce(F.col("_freq_pais_idioma"), F.lit(0)) <= F.col("_freq_entidade"))   # remove países/idiomas deslocados
    .drop("_freq_entidade", "_freq_pais_idioma"))

# Grafia canônica por nome (mais frequente, priorizando caixa mista original,
# ex.: 'Kevin Hart' vence 'KEVIN HART')
janela_grafia = Window.partitionBy("_chave").orderBy(F.col("_caixa_mista").desc(), F.col("_freq").desc(), F.col("_nome"))
df_grafia = (df_entidades.groupBy("_chave", "_nome", "_caixa_mista").agg(F.count("*").alias("_freq"))
    .withColumn("_rn", F.row_number().over(janela_grafia))
    .filter("_rn = 1")
    .select("_chave", F.col("_nome").alias("nome_entidade")))

df_pessoas_empresas = (df_entidades
    .join(df_grafia, "_chave")
    .select("id_filme", "nome_entidade", "tipo_entidade")
    .dropDuplicates())

salvar_silver(df_pessoas_empresas, "tb_pessoas_empresas")
tb = ler_silver("tb_pessoas_empresas")
display(tb.groupBy("tipo_entidade").agg(F.count("*").alias("vinculos"),
                                        F.countDistinct("nome_entidade").alias("entidades_unicas")))
display(tb.filter("tipo_entidade = 'Diretor'").groupBy("nome_entidade").count().orderBy(F.desc("count")).limit(10))

OK  cinedata_analytics.silver.tb_pessoas_empresas: 883,966 linhas


tipo_entidade,vinculos,entidades_unicas
Ator,545939,269069
Roteirista,125122,84777
Produtora,118066,45741
Diretor,94839,63679


nome_entidade,count
Kevin Dunn,77
Dustin Ferguson,66
Eric Appel,63
Alex Magaña,51
Tony Newton,43
David DeCoteau,42
Brian Volk-Weiss,38
Chad Payne,37
Damián Romay,37
Mark Polonia,34


In [0]:
# =============================================================================
# Resumo final da camada Silver
# =============================================================================
tabelas_silver = ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento",
                  "tb_avaliacoes_usuarios", "tb_generos", "tb_pessoas_empresas", "tb_cotacao_dolar"]
display(spark.createDataFrame([(t, ler_silver(t).count()) for t in tabelas_silver], ["tabela", "linhas"]))

tabela,linhas
tb_info_filmes,97611
tb_financeiro_filmes,99006
tb_metricas_engajamento,95115
tb_avaliacoes_usuarios,32412
tb_generos,141941
tb_pessoas_empresas,883966
tb_cotacao_dolar,8
